# Stage1 最差 126 日账户窗口半开区间修复

首个含末端过滤得到127行、无有效结果，失败收据保留。按 `account_window_alignment_repair_plan.json` 只改 trade_date 边界为 [first_trade,last_exit)，不重算主统计。

In [1]:
import os,json,hashlib
from pathlib import Path
import pandas as pd,qlib
from qlib.workflow import R
from scipy.stats import spearmanr
root=Path.cwd();card=root/'research/stage1_soxl_relative_target_information_20261023';plan=json.loads((card/'account_window_alignment_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'account_window_alignment_repair_plan.json').read_text());assert repair['frozen'] and sha(card/'source_audit.json')==plan['source_audit_sha256'] and sha(root/plan['account_component_result'])==plan['account_component_result_sha256']
summary=json.loads((card/'summary.json').read_text());components=json.loads((root/plan['account_component_result']).read_text());window=components['worst_126'];assert window['sessions']==126
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1,expression_cache=None,dataset_cache=None)
with R.uri_context(summary['recorder_uri']):matched=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id']).load_object(plan['source_object'])
assert len(matched)==944 and summary['recorder_id']==plan['source_recorder_id']
g=matched[(matched.trade_date>=pd.Timestamp(window['first_trade']))&(matched.trade_date<pd.Timestamp(window['last_exit']))]
assert len(g)==126 and g.H5_mature.all() and g.trade_date.is_unique
result={'status':'exact_account_trade_date_worst126_descriptive_alignment_passed','account_first_trade':window['first_trade'],'account_last_exit_exclusive':window['last_exit'],'matched_trade_dates':len(g),'first_signal_date':str(g.signal_date.min().date()),'last_signal_date':str(g.signal_date.max().date()),'mean_Stage1_SOXL_target_weight':float(g.target_weight.mean()),'mean_forward_relative_H5_log':float(g.relative_H5.mean()),'target_vs_forward_relative_H5_spearman':float(spearmanr(g.target_weight,g.relative_H5).statistic),'original_signal_date_named_cell_rows':next(x['mature_rows'] for x in json.loads((card/'source_audit.json').read_text())['named_regimes'] if x['regime']=='worst_126_cash_drag'),'full_primary_statistic_changed':False,'new_model_fits':0,'new_bootstraps':0,'new_native_accounts':0}
(card/'account_window_alignment_result.json').write_text(json.dumps(result,indent=2)+'\n');print(result)

[72831:MainThread](2026-10-23 03:42:04,998) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[72831:MainThread](2026-10-23 03:42:04,999) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[72831:MainThread](2026-10-23 03:42:04,999) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'status': 'exact_account_trade_date_worst126_descriptive_alignment_passed', 'account_first_trade': '2025-04-07', 'account_last_exit_exclusive': '2025-10-07', 'matched_trade_dates': 126, 'first_signal_date': '2025-04-04', 'last_signal_date': '2025-10-03', 'mean_Stage1_SOXL_target_weight': 0.1723043641289243, 'mean_forward_relative_H5_log': 0.03617238445056357, 'target_vs_forward_relative_H5_spearman': -0.18168669279564847, 'original_signal_date_named_cell_rows': 127, 'full_primary_statistic_changed': False, 'new_model_fits': 0, 'new_bootstraps': 0, 'new_native_accounts': 0}


## 中文结论

原账户 worst126 从 2025-04-07 成交到 2025-10-07 退出，贸易日筛选采用 [first_trade,last_exit)，恰 126 行；平均 SOXL 目标 0.172304，目标对后续 SOXL−SOXX H5 排序相关 -0.181687。首个含末端筛选误得 127 行，失败收据保留。只修正命名区间描述，不重选窗口、不改 940 行主统计或策略账户。